# Module 12 -Feature Leakage

## 1. What is Feature Leakage?

Definition: Feature leakage happens when information that would not be available at prediction time is accidentally used to train a machine learning model.

Leakage can make the model perform unrealistically well during testing but perform poorly when used on new real-world data.

### Hotel Booking Example

In the Hotel Bookings dataset, `reservation_status` and `reservation_status_date` contain information about what happened to the reservation after the booking process.

Using these features to predict `is_canceled` would allow the model to see information related to the outcome it is supposed to predict.

Therefore, these features should not be used as input features for a cancellation prediction model.

### Incorrect Feature Engineering → Leakage

Using `reservation_status` as a feature to predict `is_canceled`.

### Correct Feature Engineering → No Leakage

Using only information that would be available when the booking is made, such as `lead_time`, `adults`, `adr`, and stay details.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\HP\Sprint_6_Feature_Engineering_-_Feature_Selection\data\hotel_bookings.csv"
)

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
# Incorrect: post-outcome information used as a feature

df["Leaky_Feature"] = (
    df["reservation_status"].notna().astype(int)
)

df[
    [
        "reservation_status",
        "is_canceled",
        "Leaky_Feature"
    ]
].head()

,reservation_status,is_canceled,Leaky_Feature
0,Check-Out,0,1
1,Check-Out,0,1
2,Check-Out,0,1
3,Check-Out,0,1
4,Check-Out,0,1


## 2. Target Leakage

Definition: Target leakage is a specific type of feature leakage where a feature contains information derived from or highly dependent on the target variable.

In target leakage, the model receives information that directly or indirectly reveals the answer it is trying to predict.

### Hotel Booking Example

The target variable is `is_canceled`.

Using `reservation_status` as an input feature can cause target leakage because the reservation status describes the outcome of the booking.

For example, values such as `Canceled` or `Check-Out` are closely related to whether the booking was canceled.

### Incorrect Feature Engineering → Leakage

Including `reservation_status` or `reservation_status_date` in the feature set when predicting `is_canceled`.

### Correct Feature Engineering → No Leakage

Remove post-outcome features and use only information that was available when the booking was created.

In [4]:
# Incorrect: target-related information included as a feature

leaky_features = [
    "lead_time",
    "adr",
    "adults",
    "reservation_status"
]

X_leaky = df[leaky_features].copy()

X_leaky.head()

,lead_time,adr,adults,reservation_status
0,342,0.0,2,Check-Out
1,737,0.0,2,Check-Out
2,7,75.0,1,Check-Out
3,13,75.0,1,Check-Out
4,14,98.0,2,Check-Out


## 3. Train-Test Leakage

Definition: Train-Test Leakage happens when information from the test dataset accidentally influences the training process.

A common example is fitting a scaler, encoder, or other preprocessing technique on the complete dataset before splitting it into training and testing data.

### Hotel Booking Example

Suppose we want to predict `is_canceled` using numerical booking features.

### Incorrect Feature Engineering → Leakage

Standardizing the complete dataset first and then splitting it into training and testing data.

In this case, the test data influences the mean and standard deviation used during preprocessing.

### Correct Feature Engineering → No Leakage

First split the data into training and testing sets.

Then fit the scaler only on the training data and use the same fitted scaler to transform the test data.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

X = df[feature_cols].fillna(0)
y = df["is_canceled"]

# Incorrect: scaling before train-test split
scaler_wrong = StandardScaler()

X_scaled_wrong = scaler_wrong.fit_transform(X)

X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X_scaled_wrong,
    y,
    test_size=0.3,
    random_state=42
)

scaler_wrong.mean_

array([1.04011416e+02, 1.85640338e+00, 1.03886423e-01, 9.27598626e-01,
       2.50030153e+00, 1.01831122e+02, 5.71362761e-01])

## 4. Temporal Leakage

Definition: Temporal Leakage happens when future information is used to predict an outcome that should only be predicted using information available in the past.

This is especially important when working with data that has a time component.

### Hotel Booking Example

Hotel Booking data contains arrival-related date information such as:

- `arrival_date_year`
- `arrival_date_month`
- `arrival_date_week_number`
- `arrival_date_day_of_month`

When building a model, information from a future period should not be used to predict bookings from an earlier period.

### Incorrect Feature Engineering → Leakage

Randomly splitting time-based booking data can place future bookings in the training set and earlier bookings in the test set.

This allows the model to learn from information that would not have been available at the time of prediction.

### Correct Feature Engineering → No Leakage

Sort the bookings according to their arrival date and use earlier bookings for training and later bookings for testing.

In [6]:
# Create the arrival date

df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["arrival_date_month"] + "-" +
    df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

# Sort bookings by arrival date

df_sorted = df.sort_values("arrival_date")

# Use the 80th percentile date as the cutoff

cutoff = df_sorted["arrival_date"].quantile(0.8)

train_time = df_sorted[
    df_sorted["arrival_date"] <= cutoff
]

test_time = df_sorted[
    df_sorted["arrival_date"] > cutoff
]

print("Cutoff date:", cutoff)
print("Training rows:", len(train_time))
print("Testing rows:", len(test_time))

Cutoff date: 2017-04-22 00:00:00
Training rows: 95572
Testing rows: 23818


## 5. Post-Outcome Features

Definition: Post-Outcome Features are features that are only known after the outcome has already happened.

Using such features during prediction causes feature leakage because the model gets access to information that would not have been available at prediction time.

### Hotel Booking Example

The target variable is `is_canceled`.

The following columns contain information about the final reservation outcome:

- `reservation_status`
- `reservation_status_date`

These values are recorded after the reservation reaches a particular status.

### Incorrect Feature Engineering → Leakage

Using `reservation_status` or `reservation_status_date` to predict `is_canceled`.

### Correct Feature Engineering → No Leakage

Remove post-outcome features and use only booking information that was available when the reservation was created.

In [7]:
# Incorrect: post-outcome information used as a feature

post_outcome_features = [
    "reservation_status",
    "reservation_status_date"
]

df[
    post_outcome_features + ["is_canceled"]
].head()

,reservation_status,reservation_status_date,is_canceled
0,Check-Out,2015-07-01,0
1,Check-Out,2015-07-01,0
2,Check-Out,2015-07-02,0
3,Check-Out,2015-07-02,0
4,Check-Out,2015-07-03,0


## 6. Leakage Through Aggregations

Definition: Leakage Through Aggregations happens when an aggregated feature is calculated using information from the entire dataset, including data that should belong to the test set.

Examples of aggregation features include:

- Mean
- Sum
- Count
- Minimum
- Maximum

### Hotel Booking Example

Suppose we create an average `adr` for each hotel.

### Incorrect Feature Engineering → Leakage

Calculate the average ADR for each hotel using the complete dataset before splitting into training and testing data.

This allows information from the test set to influence the aggregated feature.

### Correct Feature Engineering → No Leakage

First split the data into training and testing sets.

Then calculate the hotel-level average ADR using only the training data and map those values to the test data.

In [8]:
# Incorrect: aggregation calculated using the complete dataset

hotel_adr_all = (
    df.groupby("hotel")["adr"]
    .transform("mean")
)

df["Hotel_Avg_ADR_Leaky"] = hotel_adr_all

df[
    ["hotel", "adr", "Hotel_Avg_ADR_Leaky"]
].head()

,hotel,adr,Hotel_Avg_ADR_Leaky
0,Resort Hotel,0.0,94.95293
1,Resort Hotel,0.0,94.95293
2,Resort Hotel,75.0,94.95293
3,Resort Hotel,75.0,94.95293
4,Resort Hotel,98.0,94.95293


## 7. Leakage Through Target Encoding

Definition: Target Encoding is a technique where a categorical feature is replaced with a statistic calculated from the target variable.

For example, each category can be replaced with its average target value.

Target encoding can cause leakage if the target information from the complete dataset is used before the train-test split.

### Hotel Booking Example

The `hotel` column is categorical and the target variable is `is_canceled`.

We can calculate the average cancellation rate for each hotel.

### Incorrect Feature Engineering → Leakage

Calculate the cancellation rate for each hotel using the complete dataset before splitting the data.

This allows target information from the test set to influence the encoded feature.

### Correct Feature Engineering → No Leakage

First split the data into training and testing sets.

Then calculate the cancellation rate for each hotel using only the training data and map those values to the test data.

In [9]:
# Incorrect: target encoding using the complete dataset

hotel_target_all = (
    df.groupby("hotel")["is_canceled"]
    .transform("mean")
)

df["Hotel_Target_Encoded_Leaky"] = hotel_target_all

df[
    ["hotel", "is_canceled", "Hotel_Target_Encoded_Leaky"]
].head()

,hotel,is_canceled,Hotel_Target_Encoded_Leaky
0,Resort Hotel,0,0.277634
1,Resort Hotel,0,0.277634
2,Resort Hotel,0,0.277634
3,Resort Hotel,0,0.277634
4,Resort Hotel,0,0.277634


## 8. Detecting Leakage

Definition: Detecting Leakage is the process of checking whether information that should not be available at prediction time has entered the model or feature engineering process.

Feature leakage can sometimes be difficult to identify because the feature may look useful and legitimate.

### Hotel Booking Example

If a Hotel Booking model achieves unusually high performance when predicting `is_canceled`, this should be investigated carefully.

Features such as `reservation_status` can create suspiciously strong predictions because they contain information related to the final booking outcome.

### Signs of Possible Leakage

- Unusually high model performance
- A sudden large improvement after adding one feature
- Features created using the target
- Features containing post-outcome information
- Preprocessing performed before train-test splitting
- Aggregations calculated using the complete dataset

### Incorrect Feature Engineering → Leakage

Using `reservation_status` together with normal booking features.

### Correct Feature Engineering → No Leakage

Remove post-outcome and target-derived features and evaluate the model using only legitimate booking-time information.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

feature_cols = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

X = df[feature_cols].fillna(0)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Clean model accuracy:", accuracy)

Clean model accuracy: 0.6932462238601781


## 9. Preventing Leakage

Feature leakage can be prevented by making sure that every feature is created only from information that would genuinely be available at prediction time.

### Prevention Checklist

- Never include a feature that is mathematically derived from the target.
- Split the data into training and testing sets before fitting a scaler, encoder, or aggregation.
- For time-based data, split the data according to time instead of randomly shuffling it.
- Check whether a feature is only known after the outcome occurs.
- Calculate aggregation features using only training data and then apply them to the test data.
- Calculate target encoding using only training target values.
- Investigate unusually high model performance for possible leakage.

### Hotel Booking Example

For predicting `is_canceled`, avoid using:

- `reservation_status`
- `reservation_status_date`
- Any feature directly derived from `is_canceled`
- Aggregations calculated using the complete dataset
- Target encodings calculated using training and test targets together

Use legitimate booking-time information such as:

- `lead_time`
- `adults`
- `children`
- `stays_in_weekend_nights`
- `stays_in_week_nights`
- `adr`
- `total_of_special_requests`

### Correct Feature Engineering → No Leakage

The complete workflow should be:

Raw Data  
↓  
Define Prediction Time  
↓  
Split Train/Test  
↓  
Feature Engineering on Training Data  
↓  
Apply Learned Transformations to Test Data  
↓  
Train Model  
↓  
Evaluate on Unseen Test Data

In [11]:
# Select only legitimate booking-time features

clean_features = [
    "lead_time",
    "adults",
    "children",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adr",
    "total_of_special_requests"
]

X = df[clean_features].fillna(0)
y = df["is_canceled"]

# Split first

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Fit preprocessing only on training data

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training features:", X_train.shape[1])

Training rows: 83573
Testing rows: 35817
Training features: 7
